# Aula 09 - Notebook: Motores de Inferência Forward e Backward Chaining
## Processo: Planta Industrial de Produção de Biodiesel (Transesterificação em Batelada - Grupo 5)

---

### 1. Fundamentos Matemáticos e Algorítmicos: Sistemas Baseados em Regras

Um **Sistema Especialista Baseado em Regras (Rule-Based Expert System)** é constituído por três pilares fundamentais:
1. **Base de Conhecimento ($\mathcal{R}$):** Conjunto finito de regras de produção formais estruturadas no paradigma condicional $SE \; (A_1 \land A_2 \land \dots \land A_n) \; ENTÃO \; C$, onde $A_i$ são os antecedentes (condições de ativação) e $C$ é o consequente (diagnóstico, alarme de trip ou ação corretiva).
2. **Memória de Trabalho / Base de Fatos ($\mathcal{F}$):** Conjunto dinâmico de proposições ativas no instante corrente, alimentadas diretamente pela telemetria dos sensores da planta (ISA-5.1) e pelas inferências deduzidas em tempo real.
3. **Motor de Inferência (Inference Engine):** O núcleo dedutivo formal que aplica algoritmos de raciocínio sobre $\mathcal{R}$ e $\mathcal{F}$.

Neste notebook, exploramos duas estratégias canônicas de inferência lógica:

* **Encadeamento para Frente (*Forward Chaining* — Data-Driven / Bottom-Up):**
  * **Princípio:** Inicia a partir dos **fatos conhecidos** (sensores ativados no ciclo de varredura do CLP/SCADA) e aplica sucessivamente o princípio de dedução *Modus Ponens*:
    $$\frac{A_1 \land A_2 \land \dots \land A_n, \quad (A_1 \land A_2 \land \dots \land A_n \rightarrow C)}{C}$$
  * As regras cujos antecedentes são subconjuntos dos fatos conhecidos entram no *Conjunto de Conflitos (Conflict Set)*. Uma estratégia de resolução de conflitos (baseada em **Prioridade estrita de segurança** e **Maior especificidade**) seleciona a regra a disparar, adicionando $C$ aos fatos conhecidos até alcançar o ponto fixo (*Fixed Point* — onde nenhuma nova regra pode ser disparada).

* **Encadeamento para Trás (*Backward Chaining* — Goal-Driven / Top-Down):**
  * **Princípio:** Inicia a partir de uma **meta ou hipótese** a ser comprovada (ex.: *"O reator sofreu risco crítico de explosão?"* ou *"A válvula de metóxido deve ser desarmada?"*) e busca recursivamente provar as submetas (antecedentes) necessárias.
  * Gera automaticamente a **Árvore de Justificativa (*Audit Trail / Proof Tree*)**, provendo explicabilidade forense total (*Explainable AI - XAI*) indispensável para auditoria em plantas químicas de alto risco.

--- 
## 2. Estrutura de Dados e Funções Auxiliares de Visualização

In [ ]:
from dataclasses import dataclass, field
from typing import Set, Tuple, List, Dict, Optional, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata uma lista de dicionarios em tabela ASCII pura alinhada."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

print("Funções utilitárias carregadas com sucesso.")

--- 
## 3. Arquitetura Orientada a Objetos do Motor Especialista do Grupo 5

In [ ]:
@dataclass
class RegraProducao:
    """Representa uma regra formal de produção em lógica de primeira ordem / proposicional."""
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    severidade: str      # 'CRÍTICA', 'ALTA', 'MÉDIA', 'BAIXA'
    prioridade: int      # 1 a 10 (10 = mais urgente / segurança intrínseca)
    tempo_resposta_max_s: float
    procedimento_pop: str

class BaseConhecimentoBiodiesel:
    """Catálogo e gerenciador indexado da base de regras do processo de biodiesel."""
    def __init__(self):
        self.regras: List[RegraProducao] = []
        self._indice_antecedentes: Dict[str, List[RegraProducao]] = {}
        self._indice_consequentes: Dict[str, List[RegraProducao]] = {}

    def adicionar_regra(
        self,
        id_regra: str,
        antecedentes: List[str],
        consequente: str,
        descricao: str,
        severidade: str = "ALTA",
        prioridade: int = 5,
        tempo_max_s: float = 5.0,
        pop: str = "Verificar malha operacional"
    ):
        regra = RegraProducao(
            id_regra=id_regra,
            antecedentes=set(antecedentes),
            consequente=consequente,
            descricao_diagnostico=descricao,
            severidade=severidade,
            prioridade=prioridade,
            tempo_resposta_max_s=tempo_max_s,
            procedimento_pop=pop
        )
        self.regras.append(regra)
        
        # Indexação bidirecional para buscas em O(1)
        for ant in antecedentes:
            if ant not in self._indice_antecedentes:
                self._indice_antecedentes[ant] = []
            self._indice_antecedentes[ant].append(regra)
            
        if consequente not in self._indice_consequentes:
            self._indice_consequentes[consequente] = []
        self._indice_consequentes[consequente].append(regra)

    def obter_regras_por_antecedente(self, fato_nome: str) -> List[RegraProducao]:
        return self._indice_antecedentes.get(fato_nome, [])

    def obter_regras_por_consequente(self, meta: str) -> List[RegraProducao]:
        return self._indice_consequentes.get(meta, [])

    def exportar_catalogo(self) -> List[Dict[str, Any]]:
        catalogo = []
        for r in sorted(self.regras, key=lambda x: (x.prioridade, len(x.antecedentes)), reverse=True):
            catalogo.append({
                "ID": r.id_regra,
                "Prioridade": r.prioridade,
                "Severidade": r.severidade,
                "SE (Antecedentes)": " AND ".join(sorted(r.antecedentes)),
                "ENTÃO (Consequente)": r.consequente,
                "Diagnóstico": r.descricao_diagnostico,
                "POP": r.procedimento_pop
            })
        return catalogo

class MotorInferenciaBiodiesel:
    """Motor de Inferência Híbrido com Forward e Backward Chaining."""
    def __init__(self, base_conhecimento: BaseConhecimentoBiodiesel):
        self.bc = base_conhecimento

    def forward_chaining(self, fatos_iniciais: Set[str]) -> Tuple[Set[str], List[Dict[str, Any]]]:
        """
        Algoritmo Forward Chaining (Data-Driven):
        Executa deduções contínuas até atingir ponto fixo, aplicando resolução de conflitos.
        """
        fatos_conhecidos = set(fatos_iniciais)
        historico_disparos = []
        passo = 1
        novos_fatos = True

        while novos_fatos:
            novos_fatos = False
            # Resolução de Conflitos: Prioridade DESC, Maior Cardinalidade de Antecedentes DESC
            regras_candidatas = sorted(
                self.bc.regras,
                key=lambda r: (r.prioridade, len(r.antecedentes), r.id_regra),
                reverse=True
            )

            for regra in regras_candidatas:
                if regra.antecedentes.issubset(fatos_conhecidos) and regra.consequente not in fatos_conhecidos:
                    fatos_conhecidos.add(regra.consequente)
                    historico_disparos.append({
                        "Passo": passo,
                        "Regra": regra.id_regra,
                        "Severidade": regra.severidade,
                        "Fato Inferido": regra.consequente,
                        "Diagnóstico": regra.descricao_diagnostico,
                        "Ação POP": regra.procedimento_pop
                    })
                    passo += 1
                    novos_fatos = True
                    break # Dispara uma regra por ciclo para garantir rigor na prioridade

        return fatos_conhecidos, historico_disparos

    def backward_chaining(
        self,
        meta: str,
        fatos_iniciais: Set[str],
        visitados: Optional[Set[str]] = None,
        nivel: int = 0
    ) -> Tuple[bool, List[str], Dict[str, Any]]:
        """
        Algoritmo Backward Chaining (Goal-Driven):
        Avalia recursivamente hipóteses e gera a árvore de justificativa formal.
        """
        if visitados is None:
            visitados = set()

        # Caso Base: Meta é um fato primário diretamente ativo
        if meta in fatos_iniciais:
            trilha = [f"{'  ' * nivel}[FATO CONHECIDO] Meta '{meta}' confirmada na base de fatos."]
            arvore = {"meta": meta, "status": "PROVADO_POR_FATO", "subarvores": []}
            return True, trilha, arvore

        # Prevenção contra loops infinitos / dependências circulares
        if meta in visitados:
            trilha = [f"{'  ' * nivel}[LOOP DETECTADO] Meta '{meta}' já presente na pilha de avaliação."]
            arvore = {"meta": meta, "status": "FALHA_LOOP", "subarvores": []}
            return False, trilha, arvore

        visitados.add(meta)
        regras_para_meta = self.bc.obter_regras_por_consequente(meta)

        if not regras_para_meta:
            trilha = [f"{'  ' * nivel}[FALHA] Nenhuma regra deduz '{meta}' e não é fato conhecido."]
            arvore = {"meta": meta, "status": "FALHA_SEM_REGRAS", "subarvores": []}
            visitados.remove(meta)
            return False, trilha, arvore

        trilha_completa = [f"{'  ' * nivel}[AVALIANDO META] Buscando provar '{meta}' via {len(regras_para_meta)} regra(s)..."]

        for regra in sorted(regras_para_meta, key=lambda r: r.prioridade, reverse=True):
            trilha_completa.append(f"{'  ' * (nivel+1)}-> Testando Regra [{regra.id_regra}]: SE ({' AND '.join(sorted(regra.antecedentes))}) ENTÃO {regra.consequente}")
            
            todos_antecedentes_provados = True
            subarvores = []

            for ant in sorted(regra.antecedentes):
                provado, sub_trilha, sub_arv = self.backward_chaining(
                    ant, fatos_iniciais, visitados, nivel + 2
                )
                trilha_completa.extend(sub_trilha)
                subarvores.append(sub_arv)
                if not provado:
                    todos_antecedentes_provados = False
                    trilha_completa.append(f"{'  ' * (nivel+2)}x Falha ao provar antecedente '{ant}'. Regra [{regra.id_regra}] descartada.")
                    break

            if todos_antecedentes_provados:
                trilha_completa.append(f"{'  ' * (nivel+1)}[SUCESSO] Regra [{regra.id_regra}] satisfeita! Meta '{meta}' PROVADA com sucesso.")
                arvore = {
                    "meta": meta,
                    "status": "PROVADO_POR_REGRA",
                    "regra": regra.id_regra,
                    "diagnostico": regra.descricao_diagnostico,
                    "pop": regra.procedimento_pop,
                    "subarvores": subarvores
                }
                visitados.remove(meta)
                return True, trilha_completa, arvore

        visitados.remove(meta)
        trilha_completa.append(f"{'  ' * nivel}[FALHA] Todas as tentativas para provar '{meta}' falharam.")
        arvore = {"meta": meta, "status": "FALHA_TODAS_REGRAS", "subarvores": []}
        return False, trilha_completa, arvore

print("Classes do Motor de Inferência e Base de Conhecimento instanciadas com sucesso.")

--- 
## 4. Cadastro da Base de Conhecimento Especialista da Planta de Biodiesel

Abaixo são registradas as regras industriais cobrindo todos os setores mapeados na planta do **Grupo 5**:
* **Setor 100:** Armazenamento e Preparação do Metóxido;
* **Setor 200:** Reator de Transesterificação, Controle Térmico e Dosagem;
* **Setor 300:** Decantação e Separação de Glicerina;
* **Setor 400:** Purificação, Lavagem e Armazenamento de Biodiesel;
* **Segurança Global:** Parada de Emergência ESD-100.

In [ ]:
bc_biodiesel = BaseConhecimentoBiodiesel()

# ==============================================================================
# 1. SETOR 200: REATOR DE TRANSESTERIFICAÇÃO (CRÍTICOS)
# ==============================================================================
bc_biodiesel.adicionar_regra(
    id_regra="R-01",
    antecedentes=["t_alta", "p1"],
    consequente="REACAO_RUNAWAY_REATOR",
    descricao="Exotermia Descontrolada e Sobrepressão no Reator R-200",
    severidade="CRÍTICA",
    prioridade=10,
    tempo_max_s=0.5,
    pop="POP-SIS-01: Desarme total imediato, corte de alimentação e resfriamento de emergência"
)

bc_biodiesel.adicionar_regra(
    id_regra="R-02",
    antecedentes=["REACAO_RUNAWAY_REATOR", "v_in_mix"],
    consequente="TRIP_CORTE_METOXIDO",
    descricao="Fechamento de Emergência da Válvula de Metóxido XV-202",
    severidade="CRÍTICA",
    prioridade=10,
    tempo_max_s=0.2,
    pop="POP-SIS-02: Fechar imediatamente XV-202 e bloquear dosagem de catalisador"
)

bc_biodiesel.adicionar_regra(
    id_regra="R-03",
    antecedentes=["REACAO_RUNAWAY_REATOR", "h1"],
    consequente="TRIP_DESLIGA_AQUECIMENTO",
    descricao="Desarme Forçado do Sistema de Aquecimento HT-201",
    severidade="CRÍTICA",
    prioridade=10,
    tempo_max_s=0.2,
    pop="POP-SIS-03: Desligar resistências térmicas HT-201 para evitar explosão"
)

bc_biodiesel.adicionar_regra(
    id_regra="R-04",
    antecedentes=["h1", "not_r1"],
    consequente="RISCO_CHOQUE_TERMICO",
    descricao="Aquecimento Ligado sem Resfriamento de Emergência (Violação Regra R1)",
    severidade="CRÍTICA",
    prioridade=9,
    tempo_max_s=1.0,
    pop="POP-HT-01: Desarmar HT-201 e verificar pressurização da linha CW-201"
)

# ==============================================================================
# 2. SETOR 100: ARMAZENAMENTO E PREPARAÇÃO DO METÓXIDO
# ==============================================================================
bc_biodiesel.adicionar_regra(
    id_regra="R-05",
    antecedentes=["g_alm", "m_mix"],
    consequente="RISCO_IGNICAO_METOXIDO",
    descricao="Vapores Inflamáveis de Metanol com Misturador AG-103 Ligado (Violação Regra R7)",
    severidade="CRÍTICA",
    prioridade=9,
    tempo_max_s=0.5,
    pop="POP-SST-04: Desligar AG-103, acionar exaustão de emergência e alarme de evacuação"
)

# ==============================================================================
# 3. SETOR 300: DECANTAÇÃO E SEPARAÇÃO DE FASES
# ==============================================================================
bc_biodiesel.adicionar_regra(
    id_regra="R-06",
    antecedentes=["v_glic", "not_i_glic"],
    consequente="CONTAMINACAO_DRENO_GLICERINA",
    descricao="Válvula de Dreno XV-301 Aberta sem Detecção de Interface (Violação Regra R6)",
    severidade="ALTA",
    prioridade=8,
    tempo_max_s=2.0,
    pop="POP-SEP-02: Fechar XV-301 imediatamente para evitar descarte indevido de biodiesel"
)

# ==============================================================================
# 4. SETOR 400: PURIFICAÇÃO E ARMAZENAMENTO FINAL
# ==============================================================================
bc_biodiesel.adicionar_regra(
    id_regra="R-07",
    antecedentes=["b_final", "not_f_lav"],
    consequente="RISCO_CAVITACAO_LAVAGEM_INCOMPLETA",
    descricao="Bomba Final P-401 Ligada sem Fluxo de Água de Lavagem (Violação Regra R8)",
    severidade="ALTA",
    prioridade=7,
    tempo_max_s=3.0,
    pop="POP-PUR-01: Desligar bomba P-401 e restabelecer linha de água de lavagem FS-401"
)

bc_biodiesel.adicionar_regra(
    id_regra="R-08",
    antecedentes=["b_final", "l_fim"],
    consequente="TRIP_TRANSBORDAMENTO_FINAL",
    descricao="Bomba P-401 Ligada com Tanque de Armazenamento Cheio",
    severidade="ALTA",
    prioridade=8,
    tempo_max_s=1.0,
    pop="POP-STG-03: Desligar bomba P-401 e fechar válvula de entrada XV-401"
)

# ==============================================================================
# 5. INTERTRAVAMENTOS GLOBAIS DE EMERGÊNCIA (ESD-100)
# ==============================================================================
bc_biodiesel.adicionar_regra(
    id_regra="R-09",
    antecedentes=["e1"],
    consequente="PARADA_EMERGENCIA_GERAL",
    descricao="Acionamento do Botão de Parada de Emergência ESD-100",
    severidade="CRÍTICA",
    prioridade=10,
    tempo_max_s=0.1,
    pop="POP-ESD-01: Desarme geral de segurança e travamento de todos os atuadores"
)

bc_biodiesel.adicionar_regra(
    id_regra="R-10",
    antecedentes=["PARADA_EMERGENCIA_GERAL", "m_reator"],
    consequente="TRIP_AGITADOR_REATOR",
    descricao="Desarme Forçado do Agitador AG-201 por Emergência Ativa (Violação Regra R5)",
    severidade="CRÍTICA",
    prioridade=10,
    tempo_max_s=0.2,
    pop="POP-ESD-02: Cortar alimentação do inversor do agitador AG-201"
)

print("=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (BIODIESEL - GRUPO 5) ===")
print(formatar_tabela(bc_biodiesel.exportar_catalogo()))

--- 
## 5. Simulação e Diagnóstico via Forward Chaining (Data-Driven)

Abaixo são simulados cenários industriais de emergência. O motor parte dos sinais discretizados dos sensores e infere em cascata todos os diagnósticos e comandos de trip de segurança.

In [ ]:
motor = MotorInferenciaBiodiesel(bc_biodiesel)

# ==============================================================================
# CENÁRIO 1: Runaway Térmico no Reator de Transesterificação (Setor 200)
# Telemetria: Alta Temperatura (t_alta), Sobrepressão (p1), Válvula de Metóxido Aberta (v_in_mix), Aquecedor Ligado (h1)
# ==============================================================================
fatos_c1 = {"t_alta", "p1", "v_in_mix", "h1"}
fatos_finais_c1, trilha_c1 = motor.forward_chaining(fatos_c1)

print("=======================================================================")
print("CENÁRIO 1: TRILHA DE DISPAROS FORWARD CHAINING (RUNAWAY REATOR R-200)")
print("=======================================================================")
print(formatar_tabela(trilha_c1))
print(f"\nFatos Finais na Memória de Trabalho ({len(fatos_finais_c1)}): {sorted(fatos_finais_c1)}")

# ==============================================================================
# CENÁRIO 2: Vazamento de Vapores Inflamáveis no Setor 100 (Metóxido)
# Telemetria: Detector de Gás Ativo (g_alm), Misturador de Metóxido Ligado (m_mix)
# ==============================================================================
fatos_c2 = {"g_alm", "m_mix"}
fatos_finais_c2, trilha_c2 = motor.forward_chaining(fatos_c2)

print("\n=======================================================================")
print("CENÁRIO 2: TRILHA DE DISPAROS FORWARD CHAINING (RISCO DE EXPLOSÃO SETOR 100)")
print("=======================================================================")
print(formatar_tabela(trilha_c2))

# ==============================================================================
# CENÁRIO 3: Emergência Geral (ESD-100) com Agitador em Operação
# Telemetria: Botão de Emergência (e1), Agitador do Reator (m_reator)
# ==============================================================================
fatos_c3 = {"e1", "m_reator"}
fatos_finais_c3, trilha_c3 = motor.forward_chaining(fatos_c3)

print("\n=======================================================================")
print("CENÁRIO 3: TRILHA DE DISPAROS FORWARD CHAINING (PROPAGAÇÃO DE ESD)")
print("=======================================================================")
print(formatar_tabela(trilha_c3))

--- 
## 6. Auditoria Forense e Prova de Metas via Backward Chaining (Goal-Driven)

No Backward Chaining, o operador ou sistema de auditoria interroga o motor com uma pergunta formal (meta de segurança). O motor constrói a árvore de prova dedutiva identificando a cadeia causal.

In [ ]:
# ==============================================================================
# AUDITORIA 1: Investigação da Meta 'TRIP_CORTE_METOXIDO'
# Sob o estado de contingência do Cenário 1
# ==============================================================================
meta_1 = "TRIP_CORTE_METOXIDO"
sucesso_bc1, trilha_bc1, arvore_bc1 = motor.backward_chaining(meta_1, fatos_c1)

print("=======================================================================")
print(f"AUDITORIA FORENSE 1: BACKWARD CHAINING PARA A META '{meta_1}'")
print("=======================================================================")
for linha in trilha_bc1:
    print(linha)
print(f"\nResultado da Prova Dedutiva: {'META PROVADA (VERDADEIRA)' if sucesso_bc1 else 'FALHA NA PROVA'}")

# ==============================================================================
# AUDITORIA 2: Investigação de Contaminação no Decantador (Setor 300)
# Telemetria: Válvula de dreno aberta (v_glic) e falha de detecção de interface (not_i_glic)
# ==============================================================================
fatos_c4 = {"v_glic", "not_i_glic"}
meta_2 = "CONTAMINACAO_DRENO_GLICERINA"
sucesso_bc2, trilha_bc2, arvore_bc2 = motor.backward_chaining(meta_2, fatos_c4)

print("\n=======================================================================")
print(f"AUDITORIA FORENSE 2: BACKWARD CHAINING PARA A META '{meta_2}'")
print("=======================================================================")
for linha in trilha_bc2:
    print(linha)
print(f"\nResultado da Prova Dedutiva: {'META PROVADA (VERDADEIRA)' if sucesso_bc2 else 'FALHA NA PROVA'}")

# ==============================================================================
# AUDITORIA 3: Teste de Hipótese Falsa / Refutação
# Testando se houve 'RISCO_IGNICAO_METOXIDO' sob o Cenário 1 (onde só o reator falhou)
# ==============================================================================
meta_3 = "RISCO_IGNICAO_METOXIDO"
sucesso_bc3, trilha_bc3, arvore_bc3 = motor.backward_chaining(meta_3, fatos_c1)

print("\n=======================================================================")
print(f"AUDITORIA FORENSE 3: PROVA NEGATIVA PARA A META '{meta_3}'")
print("=======================================================================")
for linha in trilha_bc3:
    print(linha)
print(f"\nResultado da Prova Dedutiva: {'META PROVADA' if sucesso_bc3 else 'HIPÓTESE REFUTADA COM SUCESSO (FALSA)'}")

--- 
## 7. Bateria de Testes Formais e Validação de Asserções (`assert`)

Garantia formal de conformidade dos algoritmos de inferência e das regras de segurança.

In [ ]:
# Validação de Consistência da Base de Conhecimento
assert len(bc_biodiesel.regras) == 10, "A base de regras deve conter exatamente 10 regras cadastradas."
assert len(bc_biodiesel.obter_regras_por_antecedente("t_alta")) >= 1, "Índice de antecedentes deve conter 't_alta'."
assert len(bc_biodiesel.obter_regras_por_consequente("TRIP_CORTE_METOXIDO")) >= 1, "Índice reverso de consequentes deve indexar 'TRIP_CORTE_METOXIDO'."

# Validação do Cenário 1 (Runaway)
assert "REACAO_RUNAWAY_REATOR" in fatos_finais_c1, "Cenário 1 deve inferir 'REACAO_RUNAWAY_REATOR'."
assert "TRIP_CORTE_METOXIDO" in fatos_finais_c1, "Cenário 1 deve inferir 'TRIP_CORTE_METOXIDO'."
assert "TRIP_DESLIGA_AQUECIMENTO" in fatos_finais_c1, "Cenário 1 deve inferir 'TRIP_DESLIGA_AQUECIMENTO'."

# Validação do Cenário 2 (Gás no Metóxido)
assert "RISCO_IGNICAO_METOXIDO" in fatos_finais_c2, "Cenário 2 deve inferir 'RISCO_IGNICAO_METOXIDO'."

# Validação do Cenário 3 (ESD)
assert "PARADA_EMERGENCIA_GERAL" in fatos_finais_c3, "Cenário 3 deve inferir 'PARADA_EMERGENCIA_GERAL'."
assert "TRIP_AGITADOR_REATOR" in fatos_finais_c3, "Cenário 3 deve inferir 'TRIP_AGITADOR_REATOR'."

# Validação do Backward Chaining
assert sucesso_bc1 is True, "Backward Chaining deve provar 'TRIP_CORTE_METOXIDO' sob Cenário 1."
assert sucesso_bc2 is True, "Backward Chaining deve provar 'CONTAMINACAO_DRENO_GLICERINA' sob Cenário 4."
assert sucesso_bc3 is False, "Backward Chaining deve refutar 'RISCO_IGNICAO_METOXIDO' sob Cenário 1."

print("[OK] Todos os 10 testes de inferência (Forward & Backward Chaining) foram validados com 100% de sucesso!")